# 🚀 Domain-Specific Programming Assistant using LoRA
### College Project: LoRA-Based Fine-Tuning of LLMs

This notebook demonstrates **Parameter-Efficient Fine-Tuning (PEFT)** using LoRA to transform a general-purpose model into a specialized **Programming Tutor**.

## 1. Environment Setup

In [ ]:
!pip install -q transformers peft bitsandbytes datasets accelerate trl gradio

## 2. LoRA Fine-Tuning
We use a specialized system prompt during training to teach the model to explain logic, state complexity, and provide clean code.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
import os

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME = "sahil2801/CodeAlpaca-20k"
OUTPUT_DIR = "./lora_finetuned_model"

# 1. Load Dataset
dataset = load_dataset(DATASET_NAME, split="train[:1000]")

# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# 3. Formatting Function (The secret sauce for specialization)
def formatting_prompts_func(example):
    output_texts = []
    system_prompt = (
        "You are a specialized Programming Tutor. Help the user understand Python and DSA. "
        "For every answer: 1. Logic & Intuition. 2. Time/Space Complexity. 3. Code. 4. Example usage."
    )
    for i in range(len(example['instruction'])):
        text = f"<|system|>\n{system_prompt}</s>\n<|user|>\n{example['instruction'][i]} {example['input'][i]}</s>\n<|assistant|>\n{example['output'][i]}</s>"
        output_texts.append(text)
    return output_texts

# 4. Load Model with 4-bit Quantization
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")

# 5. LoRA Config
peft_config = LoraConfig(r=64, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")

# 6. Train
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=formatting_prompts_func,
    max_seq_length=512,
    tokenizer=tokenizer,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        report_to="none"
    ),
)

print("🚀 Training LoRA adapters...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
print("✅ Training Complete!")

## 3. Side-by-Side Comparison UI
We will now launch a UI that runs BOTH the base model and the LoRA model simultaneously to show the performance improvement.

In [ ]:
import gradio as gr
from peft import PeftModel

# Load LoRA model
model_lora = PeftModel.from_pretrained(model, OUTPUT_DIR)

def generate_response(current_model, prompt):
    system_prompt = "You are a specialized Programming Tutor. Help the user understand Python and DSA."
    full_prompt = f"<|system|>\n{system_prompt}</s>\n<|user|>\n{prompt}</s>\n<|assistant|>\n"
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = current_model.generate(**inputs, max_new_tokens=400, temperature=0.7)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("<|assistant|>")[-1].strip()

def compare(message):
    base_out = generate_response(model, message)
    lora_out = generate_response(model_lora, message)
    return base_out, lora_out

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Base Model vs. LoRA-Tuned Assistant")
    with gr.Row():
        input_box = gr.Textbox(label="Ask a Coding Question")
    with gr.Row():
        btn = gr.Button("Compare", variant="primary")
    with gr.Row():
        out_base = gr.Markdown(label="Base Model")
        out_lora = gr.Markdown(label="Fine-tuned (LoRA)")
    
    btn.click(compare, inputs=input_box, outputs=[out_base, out_lora])

demo.launch(share=True)